# 03 Gold Tables (B): Cədvəllərin qurulması

Bu notebook-da `silver` cədvəllərindən gold cədvəlləri qururuq: 5 ölçü/fakt cədvəli və 2 mart cədvəli.

## Cədvəllər və birləşmələr
| Cədvəl | Hər sətir | Birləşmələr |
|---|---|---|
| `dim_geography` | bir zip kod | birləşmə yoxdur |
| `dim_products` | bir məhsul | `category_translation` (LEFT JOIN): ingilis kateqoriya adı |
| `dim_sellers` | bir satıcı | birləşmə yoxdur |
| `dim_date` | bir gün | bayramlar, məzənnə, São Paulo havası (LEFT JOIN) |
| `fact_reviews` | bir sifariş | birləşmə yoxdur, rəylər sifariş üzrə yığılır |
| `mart_seller_performance` | bir satıcı | sifariş sətirləri + sifarişlər + rəylər + satıcılar |
| `mart_daily_sales_weather` | bir gün (yalnız SP) | `dim_date` + gündəlik sifarişlər |

## Qaydalar
- Birləşmələr yalnız gold-da edilir, silver-də yox.
- "Bir-çox" əlaqəli cədvəl birləşdirilməzdən əvvəl hər açar üçün 1 sətrə yığılır.
- Hava göstəriciləri yalnız São Paulo üçündür.

## Qurulma ardıcıllığı
`dim_geography` → `dim_products`, `dim_sellers` → `dim_date` → `fact_reviews` → mart cədvəllər


In [1]:
import pandas as pd
from sqlalchemy import text
import sys

import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.append(str(_root / "notebooks"))

from db import get_engine

engine = get_engine()

## 1. `gold.dim_geography`

#Unikal poct prefiksi üçün bir sətrdə digər sütunları qruplaşdırırıq təkrarlanan məlumatları ümümləşdiririk

In [2]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_geography"))

    # Hər poçt prefiksi üçün bir sətir yaradırıq
    conn.execute(text("""
        SELECT
            zip_code_prefix,       -- Poçt prefiksi
            AVG(lat) AS lat,        -- Orta enlik koordinatı
            AVG(lng) AS lng,        -- Orta uzunluq koordinatı
            MAX(city) AS city,      -- Şəhər adı
            MAX(state) AS state     -- Ştat kodu
        INTO gold.dim_geography    -- Yeni Gold cədvəli yaradırıq
        FROM silver.geolocation   
        GROUP BY zip_code_prefix  
    """))

In [3]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT zip_code_prefix) AS unique_prefixes
    FROM gold.dim_geography
""", engine)

print(df.to_string(index=False))

 total_rows  unique_prefixes
      19015            19015


## 2. `gold.dim_products`

Burada məhsul məlumatları ilə tərcümə olunmuş kateqoriya adlarını birləşdirib bir cədvələ yığırıq

In [4]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_products"))

    conn.execute(text("""
        SELECT
            p.product_id,
            p.category_name,
            COALESCE(t.category_name_english, p.category_name, 'unknown') AS category_name_english,
            p.name_length,
            p.description_length,
            p.photos_qty,
            p.weight_g,
            p.length_cm,
            p.height_cm,
            p.width_cm
        INTO gold.dim_products
        FROM silver.products p
        LEFT JOIN silver.category_translation t
            ON p.category_name = t.category_name
    """))

In [5]:
# Sətir sayını və təkrarlanan məhsul ID-lərini yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT product_id) AS unique_products
    FROM gold.dim_products
""", engine)

df

,total_rows,unique_products
0,32951,32951


## 3. `gold.dim_sellers`

In [6]:
# Satıcı məlumatlarını Gold cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_sellers"))

    conn.execute(text("""
        SELECT
            seller_id,
            zip_code_prefix,
            city,
            state
        INTO gold.dim_sellers
        FROM silver.sellers
    """))

In [7]:
# Satıcı sayını və təkrarlanan ID-ləri yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers
    FROM gold.dim_sellers
""", engine)

df

,total_rows,unique_sellers
0,3095,3095


## 4. `gold.dim_date`

In [8]:
# Tarixlər cədvəlini yaradırıq
with engine.begin() as conn:  # Verilənlər bazasına qoşuluruq və əməliyyatları idarə edirik
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_date"))  # Əvvəlki cədvəl varsa, silirik

    conn.execute(text("""
        WITH dates AS (  -- Tarixləri ardıcıl yaratmaq üçün müvəqqəti sorğu hazırlayırıq
            SELECT CAST('2016-01-01' AS DATE) AS [date]  -- Başlanğıc tarixini 1 yanvar 2016 təyin edirik

            UNION ALL  -- Yeni tarixləri əvvəlki nəticəyə əlavə edirik

            SELECT DATEADD(DAY, 1, [date])  -- Əvvəlki tarixə 1 gün əlavə edirik
            FROM dates  -- Əvvəl yaratdığımız tarixlərdən istifadə edirik
            WHERE [date] < '2018-12-31'  -- Son tarixə çatana qədər davam edirik
        )
        SELECT
            [date],  -- Tarixi göstəririk
            YEAR([date]) AS [year],  -- Tarixin ilini çıxarırıq
            DATEPART(QUARTER, [date]) AS [quarter],  -- İlin hansı rübü olduğunu tapırıq
            MONTH([date]) AS [month],  -- Ayın nömrəsini çıxarırıq
            DATENAME(MONTH, [date]) AS month_name,  -- Ayın adını çıxarırıq
            DATEPART(ISO_WEEK, [date]) AS [week],  -- ISO standartına görə həftənin nömrəsini tapırıq
            ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 AS day_of_week,  -- Həftənin gününü bazar ertəsi=1, bazar=7 formasına gətiririk
            CASE
                WHEN ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 IN (6, 7)  -- Gün şənbə və ya bazardırsa
                THEN 1 ELSE 0  -- Həftəsonudursa 1, deyilsə 0 yazırıq
            END AS is_weekend  -- Bu sütunun adını is_weekend qoyuruq
        INTO gold.dim_date  -- Nəticəni gold.dim_date adlı yeni cədvələ yazırıq
        FROM dates  -- Yaratdığımız tarix siyahısından məlumat götürürük
        OPTION (MAXRECURSION 0)  -- Təkrarlanan sorğu üçün maksimum addım məhdudiyyətini götürürük
    """))

In [9]:
# Bayram günlərini göstərən sütunu cədvələ əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD is_holiday BIT NULL
    """))

In [10]:
# Milli bayramları tarix cədvəli ilə uyğunlaşdırırıq
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET is_holiday = CASE
            WHEN h.holiday_date IS NOT NULL THEN 1
            ELSE 0
        END
        FROM gold.dim_date d
        LEFT JOIN silver.holidays h
            ON d.[date] = h.holiday_date
            AND h.is_national = 1
    """))

Dollar məzənnəsini saxlamaq üçün sütun əlavə edirik

In [11]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD usd_per_brl DECIMAL(10,5) NULL
    """))

Hər tarix üçün həmin günə və ya əvvəlki ən yaxın məzənnəni götürürük

In [12]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET usd_per_brl = (
            SELECT TOP 1 e.rate_usd
            FROM silver.exchange_rates e
            WHERE e.rate_date <= d.[date]
            ORDER BY e.rate_date DESC
        )
        FROM gold.dim_date d
    """))

Hava məlumatlarını saxlamaq üçün 3 sütun əlavə edirik

In [13]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD sp_temp_max FLOAT NULL,
            sp_temp_min FLOAT NULL,
            sp_precipitation FLOAT NULL
    """))

Hava məlumatlarını tarixlərə uyğunlaşdırıb Gold cədvəlinə yazırıq

In [14]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET
            sp_temp_max = w.temp_max,
            sp_temp_min = w.temp_min,
            sp_precipitation = w.precipitation
        FROM gold.dim_date d
        LEFT JOIN silver.weather w
            ON d.[date] = w.weather_date
    """))

In [15]:
df = pd.read_sql("""
    SELECT TOP 10
        [date],
        is_weekend,
        is_holiday,
        usd_per_brl,
        sp_temp_max,
        sp_temp_min,
        sp_precipitation
    FROM gold.dim_date
    ORDER BY [date]
""", engine)

df

,date,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation
0,2016-01-01,0,True,0.25250,27.7,21.0,8.8
1,2016-01-02,1,False,0.25250,24.5,19.8,26.0
2,2016-01-03,1,False,0.25250,23.2,18.9,4.4
3,2016-01-04,0,False,0.24755,25.4,17.7,0.0
4,2016-01-05,0,False,0.24977,26.4,16.8,0.0
5,2016-01-06,0,False,0.24791,28.5,18.5,0.0
6,2016-01-07,0,False,0.24699,28.6,18.6,0.3
7,2016-01-08,0,False,0.24862,28.4,20.0,0.9
8,2016-01-09,1,False,0.24862,28.9,20.4,30.1
9,2016-01-10,1,False,0.24862,27.4,21.5,14.2


## 5. `gold.fact_reviews`

In [16]:
df = pd.read_sql("SELECT TOP 5 * FROM silver.reviews", engine)
df

,review_id,order_id,review_score,comment_title,comment_message,creation_date,answer_timestamp
0,00041b1f51e77a663df6db3f539d694f,998b1beee068744ac833dd8b31944623,1,None,None,2018-05-24 00:00:00,2018-05-24 12:45:38
1,0005bc906dfe3b73261020bf8e9b76cf,75cc0defdf90cda65066ce0ac171e2ab,5,None,Muito bom,2017-10-15 01:00:00,2017-10-15 17:53:22
2,0006cce9fd63c8ea1c488b88d4cb3f02,df940e48580d0f2c2283d0782c7f99e3,5,None,None,2017-08-26 00:00:00,2017-08-30 12:03:31
3,000da26f546e5d51b6336486f57fc632,dc6a29077e547f5fe4377f251fd1c5d0,5,None,"Muito satisfeita com o produto, entrega e a pr...",2017-06-03 00:00:00,2017-06-07 04:00:21
4,00112b1cb3cd58339c91aad88df2b835,c7068b2d09a51ca53dd0a06c2526be34,4,Entrega muito rápida,Nota 10 pela entrega.\r\nO produto não tem a c...,2018-05-03 00:00:00,2018-05-03 22:14:11


Bu hissədə rəylər cədvəlində ümumi baxış olaraq neçə sətr,neçə distinct rəy id,null olan id sütunlar cəmi , min və max rəy scorlar göstərilib  

In [17]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT review_id) AS unique_reviews,
        SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS null_review_id,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS null_review_score,
        MIN(review_score) AS min_score,
        MAX(review_score) AS max_score
    FROM silver.reviews
""", engine)

display(df)

,total_rows,unique_reviews,null_review_id,null_order_id,null_review_score,min_score,max_score
0,99224,98410,0,0,0,1,5


In [18]:
# Rəy məlumatlarını Gold fakt cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_reviews"))

    conn.execute(text("""
        SELECT
            r.review_id,
            r.order_id,
            r.review_score,
            r.comment_title,
            r.comment_message,
            r.creation_date,
            r.answer_timestamp
        INTO gold.fact_reviews
        FROM silver.reviews r
        INNER JOIN silver.orders o
            ON r.order_id = o.order_id
    """))

In [19]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(review_id, order_id)) AS unique_combinations
    FROM gold.fact_reviews
""", engine)

display(df)

,total_rows,unique_combinations
0,99224,99224


## 6. `gold.mart_seller_performance`

In [20]:
df = pd.read_sql("SELECT TOP 5 * FROM silver.order_items", engine)
display(df)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [21]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_seller_id,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS null_freight
    FROM silver.order_items
""", engine)

df

,total_rows,unique_orders,unique_sellers,null_order_id,null_seller_id,null_price,null_freight
0,112650,98666,3095,0,0,0,0


In [22]:
# Satıcı performansı mart cədvəlini yaradırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_seller_performance"))

    conn.execute(text("""
        SELECT
            s.seller_id,
            s.zip_code_prefix,
            s.city,
            s.state,
            COALESCE(x.total_orders, 0) AS total_orders,
            COALESCE(x.items_sold, 0) AS items_sold,
            COALESCE(x.total_sales, 0) AS total_sales,
            COALESCE(x.total_freight, 0) AS total_freight
        INTO gold.mart_seller_performance
        FROM silver.sellers s
        LEFT JOIN (
            SELECT
                oi.seller_id,
                COUNT(DISTINCT oi.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.order_items oi
            INNER JOIN silver.orders o
                ON oi.order_id = o.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY oi.seller_id
        ) x
            ON s.seller_id = x.seller_id
    """))

Yeni mart cədvəlinin sətir sayını və satıcı unikallığını yoxlayırıq

In [23]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        SUM(total_orders) AS total_orders,
        SUM(items_sold) AS items_sold,
        SUM(total_sales) AS total_sales
    FROM gold.mart_seller_performance
""", engine)

display(df)

,total_rows,unique_sellers,total_orders,items_sold,total_sales
0,3095,3095,97819,110197,13221498.11


## 7. `gold.mart_daily_sales_weather`

In [24]:
df = pd.read_sql("""
    SELECT TOP 5
        order_id,
        order_status,
        order_purchase_timestamp
    FROM silver.orders
    ORDER BY order_purchase_timestamp
""", engine)

display(df)

,order_id,order_status,order_purchase_timestamp
0,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19
1,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34
2,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19
3,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38
4,71303d7e93b399f5bcd537d124c0bcfa,canceled,2016-10-02 22:07:52


In [25]:
# Gündəlik satışları tarix və hava məlumatları ilə birləşdiririk
df = pd.read_sql("""
    SELECT
        d.[date] AS sales_date,
        d.[year],
        d.[month],
        d.is_weekend,
        d.is_holiday,
        d.usd_per_brl,
        d.sp_temp_max,
        d.sp_temp_min,
        d.sp_precipitation,
        COALESCE(s.total_orders, 0) AS total_orders,
        COALESCE(s.items_sold, 0) AS items_sold,
        COALESCE(s.total_sales, 0) AS total_sales,
        COALESCE(s.total_freight, 0) AS total_freight
    FROM gold.dim_date d
    LEFT JOIN (
        SELECT
            CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
            COUNT(DISTINCT o.order_id) AS total_orders,
            COUNT(*) AS items_sold,
            SUM(oi.price) AS total_sales,
            SUM(oi.freight_value) AS total_freight
        FROM silver.orders o
        INNER JOIN silver.order_items oi
            ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY CAST(o.order_purchase_timestamp AS DATE)
    ) s
        ON d.[date] = s.sales_date
    ORDER BY d.[date]
""", engine)

df.head()

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2016-01-01,2016,1,0,True,0.25250,27.7,21.0,8.8,0,0,0.0,0.0
1,2016-01-02,2016,1,1,False,0.25250,24.5,19.8,26.0,0,0,0.0,0.0
2,2016-01-03,2016,1,1,False,0.25250,23.2,18.9,4.4,0,0,0.0,0.0
3,2016-01-04,2016,1,0,False,0.24755,25.4,17.7,0.0,0,0,0.0,0.0
4,2016-01-05,2016,1,0,False,0.24977,26.4,16.8,0.0,0,0,0.0,0.0


In [26]:
# Gündəlik satış və hava məlumatlarını Gold mart cədvəlinə yazırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_daily_sales_weather"))

    conn.execute(text("""
        SELECT
            d.[date] AS sales_date,
            d.[year],
            d.[month],
            d.is_weekend,
            d.is_holiday,
            d.usd_per_brl,
            d.sp_temp_max,
            d.sp_temp_min,
            d.sp_precipitation,
            COALESCE(s.total_orders, 0) AS total_orders,
            COALESCE(s.items_sold, 0) AS items_sold,
            COALESCE(s.total_sales, 0) AS total_sales,
            COALESCE(s.total_freight, 0) AS total_freight
        INTO gold.mart_daily_sales_weather
        FROM gold.dim_date d
        LEFT JOIN (
            SELECT
                CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
                COUNT(DISTINCT o.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.orders o
            INNER JOIN silver.order_items oi
                ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY CAST(o.order_purchase_timestamp AS DATE)
        ) s
            ON d.[date] = s.sales_date
    """))

In [27]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM gold.mart_daily_sales_weather
    ORDER BY sales_date
""", engine)

df

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2016-01-01,2016,1,0,True,0.25250,27.7,21.0,8.8,0,0,0.0,0.0
1,2016-01-02,2016,1,1,False,0.25250,24.5,19.8,26.0,0,0,0.0,0.0
2,2016-01-03,2016,1,1,False,0.25250,23.2,18.9,4.4,0,0,0.0,0.0
3,2016-01-04,2016,1,0,False,0.24755,25.4,17.7,0.0,0,0,0.0,0.0
4,2016-01-05,2016,1,0,False,0.24977,26.4,16.8,0.0,0,0,0.0,0.0
